# Improve a support agent's playbook

Our support assistant sends refund and login requests to the wrong team.
Its instructions live in a **playbook**: a small collection of named rules.
We'll use its mistakes to improve one rule at a time, keeping the others intact.

**Start with rules → revise one rule → test six tickets → inspect the change.**

The [skill lesson](https://sentient-xyz.github.io/meta-evolve-docs/guides/skill-composition/) stores instructions in one document.
Here, naming each rule makes individual changes easy to inspect. This five-cell
lesson uses the same six tickets and responder, with every definition below.
The responder and revisions are simple Python simulations; no model or API key
is needed.



## 1. Install

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

## 2. Start with three rules and six tickets

`PLAYBOOK_SEED` maps each rule's name to its instruction text. The names stay
the same when we revise the text. `TICKETS` lists the expected team for each
request; these answers stay fixed throughout the run.

Our simulated assistant already handles invoices and passwords. It handles
refunds and login requests only when those words appear in its instructions.
It does not understand arbitrary wording or rule priority.

In [ ]:
import meta_evolve as meta

# Give each rule a name so we can change it without replacing the others.
PLAYBOOK_SEED = {
    "billing": "Route invoices to billing.",
    "account": "Route passwords to account.",
    "technical": "Route everything else to technical.",
}
# These expected teams stay fixed while the instructions change.
TICKETS = (
    ("Please send my invoice", "billing"),
    ("I need a refund", "billing"),
    ("I forgot my password", "account"),
    ("My login is blocked", "account"),
    ("The dashboard freezes", "technical"),
    ("Export is broken", "technical"),
)


def routing_response(instructions, ticket):
    # A simulation: refunds and login need matching words in the instructions.
    text = ticket.lower()
    if "invoice" in text:
        return "billing"
    if "password" in text:
        return "account"
    if "refund" in instructions and "refund" in text:
        return "billing"
    if "login" in instructions and "login" in text:
        return "account"
    return "technical"

## 3. Test the answers and record mistakes

`evaluate_playbook` joins the rule text and gives it to the assistant. It checks
each answer against the expected team, then records the score and mistakes.
Meta-Evolve calls these recorded observations **evidence**.

For the starting rules, “I need a refund” goes to `technical` instead of
`billing`. The recorded mistake contains the ticket and both answers.

In [ ]:
def evaluate_playbook(playbook):
    instructions = "\n".join(playbook.values())  # The assistant reads all the rules.
    misses = []
    for ticket, expected in TICKETS:
        actual = routing_response(instructions, ticket)
        if actual != expected:
            misses.append({"ticket": ticket, "expected": expected, "actual": actual})
    # Keep the wrong answers as well as the fraction correct.
    return meta.EvaluationResult(
        metrics={"score": (len(TICKETS) - len(misses)) / len(TICKETS)},
        evidence=(meta.EvidenceDraft(kind="routing-misses", data={"misses": misses}),),
    )

## 4. Amend the rule behind one mistake

`revise_playbook` reads the latest shared mistakes and starts with the first.
The expected team chooses the rule: a missed refund updates `billing` by
adding “Route requests like 'I need a refund' to billing.”

`{**playbook, ...}` builds a new mapping with that one replacement. The other
rules and the previous version remain intact. Missing feedback reports an
error; an empty mistake list leaves the rules unchanged.

In [ ]:
def revise_playbook(playbook, *, context):
    feedback = context.evidence.latest("routing-misses")
    if feedback is None:
        raise ValueError("No routing feedback was selected.")
    misses = feedback.data["misses"]
    if not misses:
        return playbook  # Nothing to fix.
    miss = misses[0]  # Use one recorded mistake to choose the next change.
    team = miss["expected"]
    correction = f"Route requests like {miss['ticket']!r} to {team}."
    # Return a new version with just this team's rule amended.
    return {**playbook, team: playbook[team] + " " + correction}

## 5. Run, inspect, and keep the rules

`trials=2` allows two revisions after checking the starting playbook.
`RecentAncestors()` shares recent recorded mistakes with the proposer;
recording them alone does not make them available as feedback.

Each revision is tested on the same six tickets. Meta-Evolve keeps the version
with the better score. The output shows which rule changed, its exact edit,
and the tickets that still need attention.

In [ ]:
playbook_result = meta.improve(
    seed=PLAYBOOK_SEED, proposer=revise_playbook, evaluator=evaluate_playbook,
    trials=2,  # Two revisions after checking the starting rules.
    context=meta.RecentAncestors(),  # Share recent mistakes with the proposer.
)

# Compare each version with the one before it.
previous = PLAYBOOK_SEED
for number, version in enumerate(playbook_result.trials()):
    rules = version.artifact.value
    missed = [case["ticket"] for case in version.evidence[0].data["misses"]]
    print(f"Version {number}: {version.metrics['score']:.0%}; misrouted: {missed}")
    for name, text in rules.items():
        if text != previous[name]:
            print(f"  {name}: {previous[name]}\n    -> {text}")
    previous = rules

# Keep these rules to use with your assistant.
selected_playbook = playbook_result.best().value
print("Selected playbook:")
for name, text in selected_playbook.items():
    print(f"  {name}: {text}")
# Output:
# Version 0: 67%; misrouted: ['I need a refund', 'My login is blocked']
# Version 1: 83%; misrouted: ['My login is blocked']
#   billing: Route invoices to billing.
#     -> Route invoices to billing. Route requests like 'I need a refund' to billing.
# Version 2: 100%; misrouted: []
#   account: Route passwords to account.
#     -> Route passwords to account. Route requests like 'My login is blocked' to account.
# Selected playbook:
#   account: Route passwords to account. Route requests like 'My login is blocked' to account.
#   billing: Route invoices to billing. Route requests like 'I need a refund' to billing.
#   technical: Route everything else to technical.

The refund mistake changes `billing`; the login mistake changes `account`.
`technical` stays the same. `selected_playbook` holds the resulting named rules;
`"\n".join(selected_playbook.values())` gives you the instruction text to pass
to your assistant. The original `PLAYBOOK_SEED` remains unchanged.

These scores measure our simulated assistant on the six tickets used to choose
revisions. They do not measure a real model or performance on unseen requests.

## Change and predict

1. Change `trials=2` to `trials=1` and rerun the last cell. Only `billing`
   changes: **83%**, with login still misrouted.
2. In cell 2, change the starting billing rule to
   `"Route invoices and refunds to billing."`.
3. Run cells 2–5 again, keeping `trials=1`. The starting score is now **83%**;
   the one revision changes `account` and reaches **100%**. The billing rule
   you supplied stays intact.

The next change follows the recorded mistake, so a different starting rule
changes which revision is useful. Set `trials=0` to test only your starting rules.

To use your own task, replace `TICKETS` with labeled examples and
`routing_response` with your assistant. Replace the instruction-building part
of `revise_playbook` with your revision function, including the selected
feedback in its request. [Use your model SDK](https://sentient-xyz.github.io/meta-evolve-docs/guides/providers/) shows the connection.

## Go further

For a few rules, a handwritten loop is reasonable. Meta-Evolve keeps each
version and its mistakes, selects which feedback reaches the proposer, and
limits the number of revisions. [Save and reopen a run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/)
shows how to retain that history after closing your notebook.

[Choose what to try next](https://sentient-xyz.github.io/meta-evolve-docs/learn/02-change-search/) compares Greedy and
TreeSearch through their recorded parent choices. Search chooses which version
to revise; this lesson's feedback guides the rule edit.

[The advanced ACE study](https://sentient-xyz.github.io/meta-evolve-docs/research/ace/) explores a larger itemized playbook,
with separate development, consolidation, and final checks. This small lesson
demonstrates revising a named rule from a recorded mistake; the study explains
how its implementation relates to ACE.

[Previous: Improve a support agent's skill](https://sentient-xyz.github.io/meta-evolve-docs/guides/skill-composition/) ·
[Improve an agent's harness](https://sentient-xyz.github.io/meta-evolve-docs/guides/harness-evolution/) · [Choose an example](https://sentient-xyz.github.io/meta-evolve-docs/examples/)